# BGE-M3 chunking ablation

Create comparable recursive chunk indexes at 300 characters, 450 tokens, and 600 tokens. Preserve article structure and Khmer punctuation, then generate BGE-M3 chunk and article embeddings from the same corpus snapshot.

In [ ]:
import gc
import json
import re
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data-pipeline').is_dir())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
INPUT_PATH = PROCESSED_DIR / 'articles_normalized.jsonl'
EMBEDDING_DIR = PROCESSED_DIR / 'test' / 'embeddings'
MODEL_NAME = 'BAAI/bge-m3'
TOKENIZER_NAME = 'aisingapore/SEA-LION-E5-Embedding-600M'
TOKENIZER = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
TOKENIZER.model_max_length = 10**9
VARIANTS = {
    'recursive_300_chars': {'limit': 300, 'unit': 'chars'},
    'recursive_450_tokens': {'limit': 450, 'unit': 'tokens'},
    'recursive_600_tokens': {'limit': 600, 'unit': 'tokens'},
}

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

articles = load_jsonl(INPUT_PATH)
unique_articles = {}
for article in articles:
    key = str(article.get('article_id') or f"{article.get('document_id')}-{article.get('article_number')}")
    if key not in unique_articles or len(article.get('text', '')) > len(unique_articles[key].get('text', '')):
        unique_articles[key] = article

def measure(text, unit):
    return len(text) if unit == 'chars' else len(TOKENIZER.encode(text, add_special_tokens=False))

def pack(units, limit, unit):
    output, current = [], ''
    for unit_text in units:
        candidate = f'{current} {unit_text}'.strip() if current else unit_text
        if current and measure(candidate, unit) > limit:
            output.append(current); current = unit_text
        else:
            current = candidate
    if current: output.append(current)
    return output

def recursive_split(text, limit, unit, level=0):
    text = ' '.join(str(text).split())
    if not text: return []
    if measure(text, unit) <= limit: return [text]
    separators = [r'\n\n+', r'\n+', r'(?<=[។៕?!])\s+', '\u200b', r'\s+']
    if level < len(separators):
        units = [part.strip() for part in re.split(separators[level], text) if part.strip()]
        if len(units) > 1:
            packed = pack(units, limit, unit)
            return [piece for item in packed if measure(item, unit) <= limit for piece in recursive_split(item, limit, unit, level + 1)]
        return recursive_split(text, limit, unit, level + 1)
    pieces, current = [], ''
    for char in text:
        candidate = current + char
        if current and measure(candidate, unit) > limit:
            pieces.append(current); current = char
        else: current = candidate
    if current: pieces.append(current)
    return pieces

def context(article):
    values = [article.get('document_title'), f"មាត្រា {article.get('article_number')}" if article.get('article_number') is not None else None, article.get('chapter_title'), article.get('section_title')]
    return ' | '.join(str(value).strip() for value in values if value)

def make_chunks(article, limit, unit):
    prefix = context(article)
    content_limit = max(1, limit - measure(prefix, unit))
    chunks = recursive_split(article.get('search_text', article.get('text', '')), content_limit, unit)
    return [f'{prefix}\n{chunk}'.strip() if prefix else chunk for chunk in chunks]

def write_variant(name, config, model):
    records = []
    for article in articles:
        chunks = make_chunks(article, config['limit'], config['unit'])
        for index, text in enumerate(chunks, start=1):
            records.append({'chunk_id': f"{article['id']}-{name}-{index}", 'article_id': article['article_id'], 'parent_id': article['id'], 'document_id': article['document_id'], 'article_number': article['article_number'], 'chunk_index': index, 'chunk_count': len(chunks), 'text': text, 'document_title': article.get('document_title'), 'chapter_title': article.get('chapter_title'), 'section_title': article.get('section_title'), 'source_url': article.get('source_url'), 'pdf_url': article.get('pdf_url')})
    texts = [record['text'] for record in records]
    vectors = model.encode(texts, batch_size=2, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    prefix = f'chunks_bge_m3_{name}'
    EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
    np.save(EMBEDDING_DIR / f'{prefix}.npy', vectors)
    (EMBEDDING_DIR / f'{prefix}_metadata.jsonl').write_text(''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in records), encoding='utf-8')
    manifest = {'model': MODEL_NAME, 'variant': name, 'limit': config['limit'], 'unit': config['unit'], 'article_input': str(INPUT_PATH), 'chunk_count': len(records), 'embedding_dimension': int(vectors.shape[1]), 'normalized_embeddings': True}
    (EMBEDDING_DIR / f'{prefix}_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return len(records), vectors.shape[1]

model = SentenceTransformer(MODEL_NAME)
results = {name: {'chunks': count, 'dimension': dimension} for name, config in VARIANTS.items() for count, dimension in [write_variant(name, config, model)]}
article_records = list(unique_articles.values())
article_texts = [f"{context(article)}\n{article.get('search_text', article.get('text', ''))}".strip() for article in article_records]
article_vectors = model.encode(article_texts, batch_size=2, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
np.save(EMBEDDING_DIR / 'articles_bge_m3.npy', article_vectors)
(EMBEDDING_DIR / 'articles_bge_m3_metadata.jsonl').write_text(''.join(json.dumps({'article_id': article['article_id'], 'document_id': article['document_id'], 'article_number': article['article_number']}, ensure_ascii=False) + '\n' for article in article_records), encoding='utf-8')
(EMBEDDING_DIR / 'articles_bge_m3_manifest.json').write_text(json.dumps({'model': MODEL_NAME, 'input': str(INPUT_PATH), 'article_count': len(article_records), 'embedding_dimension': int(article_vectors.shape[1]), 'normalized_embeddings': True}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps({'variants': results, 'unique_articles': len(article_records)}, ensure_ascii=False, indent=2))